## Import Packages

In [16]:
# general
import numpy as np
import pandas as pd
import os
import warnings
from tqdm.notebook import tqdm # if this line throws an error, type "conda install tqdm" in your terminal to install tqdm

# plotting
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# my functions/classes
import sys
sys.path.append("../core_scripts/")
from ECMclass import ECM

## User Inputs

In [17]:
# smoothing window, mm
window = 10

# paths
global path_to_data, path_to_figures, path_to_metadata, metadata_file
path_to_data = '../../data/'
path_to_figures = '../../figures/basic_plots/'
path_to_metadata = '../../data/'
metadata_file = 'metadata.csv'

In [18]:
# set globals
global meta, my_cmap

# Load metadata
meta = pd.read_csv(path_to_data+metadata_file)

# make colormap
my_cmap = matplotlib.colormaps['Spectral']

## Functions

In [19]:
def plotquarter(y_vec,ycor,d,meas,surface_imperfection,tephra_or_layer,rock_or_debris,axs,rescale,res = 0.002):
    
    width = y_vec[1] - y_vec[0]
    
    for y in y_vec:
        
        idx = ycor==y
        
        tmeas = meas[idx]
        tsurface_imperfection = surface_imperfection[idx]
        ttephra_or_layer = tephra_or_layer[idx]
        trock_or_debris = rock_or_debris[idx]
        tycor = ycor[idx]
        td = d[idx]

        #downsample ECM to save plotting time (as needed)
        int_lo = round(min(td),2)
        int_hi = round(max(td),2)
        depth_interp = np.linspace(int_lo,int_hi,int((int_hi-int_lo)/res)+1)
        meas_interp = np.interp(depth_interp,np.flip(td),np.flip(tmeas))
        surface_imperfection_interp = np.interp(depth_interp,np.flip(td),np.flip(tsurface_imperfection))
        tephra_or_layer_interp = np.interp(depth_interp,np.flip(td),np.flip(ttephra_or_layer))
        rock_or_debris_interp = np.interp(depth_interp,np.flip(td),np.flip(trock_or_debris))
        td = depth_interp
        tmeas = meas_interp
        tsurface_imperfection = np.round(surface_imperfection_interp)
        ttephra_or_layer = np.round(tephra_or_layer_interp)
        trock_or_debris = np.round(rock_or_debris_interp)

        x0 = y-(width-0.2)/2
        track_width = width-0.2

        # Base ECM (and surface imperfection masking) pass
        for i in range(len(tmeas)-1):
            
            if tsurface_imperfection[i] == 0:        
                axs.add_patch(Rectangle((x0,td[i]),track_width,td[i+1]-td[i],facecolor=my_cmap(rescale(tmeas[i]))))
            else:
                axs.add_patch(Rectangle((x0,td[i]),track_width,td[i+1]-td[i],facecolor='k'))

        # Overlay pass for contiguous tephra/layer and rock/debris regions
        def add_contiguous_regions(mask, draw_region):
            run_start = None
            for i, is_on in enumerate(mask):
                if is_on and run_start is None:
                    run_start = i
                elif not is_on and run_start is not None:
                    draw_region(run_start, i - 1)
                    run_start = None
            if run_start is not None:
                draw_region(run_start, len(mask) - 1)

        tephra_mask = ttephra_or_layer[:-1] == 1
        rock_mask = trock_or_debris[:-1] == 1

        def draw_tephra_region(i0, i1):
            axs.add_patch(
                Rectangle(
                    (x0, td[i0]),
                    track_width,
                    td[i1 + 1] - td[i0],
                    facecolor='none',
                    edgecolor='purple',
                    hatch='xx',
                    linewidth=0
                )
            )

        
        def draw_rock_region(i0, i1):
            axs.add_patch(
                Rectangle(
                    (x0, td[i0]),
                    track_width,
                    td[i1 + 1] - td[i0],
                    facecolor='none',
                    edgecolor='black',
                    hatch='oo',
                    linewidth=0
                )
            )

        # def draw_rock_region(i0, i1):
        #     y0 = td[i0]
        #     y1 = td[i1 + 1]
        #     y_spacing = max(res * 4, 0.008)
        #     y_pts = np.arange(y0, y1, y_spacing)
            
        #     if len(y_pts) == 0:
        #         y_pts = np.array([(y0 + y1) / 2])

        #     # 4 columns across the track (instead of 2), with slight random x jitter
        #     x_base = np.linspace(x0 + 0.15 * track_width, x0 + 0.85 * track_width, 4)
        #     xx, yy = np.meshgrid(x_base, y_pts)

        #     rng = np.random.default_rng()
        #     jitter_span = 0.04 * abs(track_width)
        #     x_min = min(x0 + 0.05 * track_width, x0 + 0.95 * track_width)
        #     x_max = max(x0 + 0.05 * track_width, x0 + 0.95 * track_width)
        #     jitter = rng.uniform(-jitter_span, jitter_span, size=xx.shape) if jitter_span > 0 else 0
        #     x_jittered = np.clip(
        #     xx + jitter,
        #     x_min,
        #     x_max
        #     )

        #     axs.scatter(
        #     x_jittered.ravel(),
        #     yy.ravel(),
        #     c='k',
        #     s=12,  # 50% smaller than previous s=24
        #     marker='o',
        #     linewidths=0,
        #     zorder=5
        #     )

        add_contiguous_regions(tephra_mask, draw_tephra_region)
        add_contiguous_regions(rock_mask, draw_rock_region)

    return()

In [20]:
def plot_basic(core):

    print(path_to_data)

    # check that path_to_figures+core exists, if not create it
    if not os.path.exists(path_to_figures+core):
        os.makedirs(path_to_figures+core)

    # filter metadata for the core of interest
    core_meta = meta[meta['core'] == core]

    # get list of unique sections for the core
    sections = core_meta['section'].unique()

    # loop though sections and plot each one
    total_sections = len(sections)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        with np.errstate(divide='ignore', invalid='ignore'):
            with tqdm(total=total_sections, desc=f"{core}: starting", unit="section", leave=True) as section_progress:
                for cnt, section in enumerate(sections, start=1):

                    # progress bar here, including update on current section
                    section_progress.set_description(f"{core} sec {cnt}/{total_sections} ({section})")

                    section_meta = core_meta[core_meta['section'] == section]

                    # get list of unique faces for the core
                    faces = section_meta['face'].unique()

                    # load ecm data for the section
                    data = []
                    for index,row in section_meta.iterrows():
                        #print("    Loading ECM data for core: {}, section: {}, face: {}, ACorDC: {}".format(row['core'],row['section'],row['face'],row['ACorDC']))

                        # check if filename exists in ../../data/clean/[core]/
                        if os.path.exists(path_to_data+'clean/'+row['core']+'/'+row['core']+'-'+row['section']+'-'+row['face']+'-'+row['ACorDC']+'.csv'):
                            path_to_file = path_to_data+'clean/'
                        else:
                            path_to_file = path_to_data+'not-cleaned/'
                        data_item = ECM(row['core'],row['section'],row['face'],row['ACorDC'],path_to_file,path_to_metadata + metadata_file)
                        data_item.rem_ends(10)
                        data_item.smooth(window)
                        data_item.norm_all()
                        #data_item.norm_outside()
                        data.append(data_item)

                    # find min and max depths
                    minvec = []
                    maxvec = []
                    for data_face in data:
                        minvec.append(min(data_face.depth))
                        maxvec.append(max(data_face.depth))
                    dmin = min(minvec)
                    dmax = max(maxvec)

                    # get AC and data rescale
                    # NOTE: NEED TO DEAL WITH BUTTON!!!
                    AC_all = []
                    DC_all = []
                    for data_face in data:
                        mask = data_face.button == 0
                        if data_face.ACorDC == 'AC':
                            AC_all.extend(data_face.meas[mask])
                        else:
                            DC_all.extend(data_face.meas[mask])
                    if len(DC_all) == 0:
                        DCpltmin, DCpltmax = np.nan, np.nan   # or sensible defaults / skip plotting
                    else:
                        DCpltmin = np.percentile(DC_all, 5)
                        DCpltmax = np.percentile(DC_all, 95)
                    if len(AC_all) == 0:
                        ACpltmin, ACpltmax = np.nan, np.nan   # or sensible defaults / skip plotting
                    else:  
                        ACpltmin = np.percentile(AC_all,5)
                        ACpltmax = np.percentile(AC_all,95)
                    ACrescale = lambda k: (k-ACpltmin) /  (ACpltmax-ACpltmin)
                    DCrescale = lambda k: (k-DCpltmin) /  (DCpltmax-DCpltmin)
        
                    # now fill in the data (calls sub-fucntion based on the cut)
                    if ['r'] in faces:
                        fig, axs = plot_tr(data,dmin,dmax,ACrescale,DCrescale)
                    elif ['l'] in faces:
                        fig, axs = plot_tl(data,dmin,dmax,ACrescale,DCrescale)
                    elif ['w2'] in faces and ['w1'] in faces:
                        fig, axs = plot_w1w2(data,dmin,dmax,ACrescale,DCrescale)
                    elif (['w1'] in faces and ['w2'] not in faces) or 'half' in faces:
                        fig, axs = plot_w1(data,dmin,dmax,ACrescale,DCrescale)

                    # finish figure
                    fig.suptitle(f"{core} section {section}")
                    fig.tight_layout()
                    plt.subplots_adjust(wspace=0)

                    # add colorbar
                    ACcbar_ax = fig.add_axes([0.07,-0.05,0.35,0.05])
                    ACnorm = matplotlib.colors.Normalize(vmin=ACpltmin,vmax=ACpltmax)
                    DCcbar_ax = fig.add_axes([0.58,-0.05,0.35,0.05])
                    DCnorm = matplotlib.colors.Normalize(vmin=DCpltmin,vmax=DCpltmax)
                    ACcbar = fig.colorbar(matplotlib.cm.ScalarMappable(norm=ACnorm, cmap=my_cmap),cax=ACcbar_ax,
                                orientation='horizontal',label='Current (amps)')
                    DCcbar = fig.colorbar(matplotlib.cm.ScalarMappable(norm=DCnorm, cmap=my_cmap),cax=DCcbar_ax,
                                orientation='horizontal',label='Current (amps)')

                    # add a key on the right for surface imperfections, tephra/layers, and rock/debris
                    key_ax = fig.add_axes([1,0.1,0.05,0.8])
                    key_ax.add_patch(Rectangle((0,0.8),1,0.1,facecolor='k'))
                    key_ax.text(1.2,0.85,'Surface Imperfection',fontsize=10,verticalalignment='center')
                    key_ax.add_patch(Rectangle((0,0.6),1,0.1,facecolor='none',edgecolor='purple',hatch='xx',linewidth=0))
                    key_ax.text(1.2,0.65,'Tephra/Layer',fontsize=10,verticalalignment='center')
                    # now add patch with black dots for rock/debris
                    key_ax.add_patch(Rectangle((0,0.4),1,0.1,facecolor='none',edgecolor='k',hatch='oo',linewidth=0))
                    key_ax.text(1.2,0.45,'Rock/Debris',fontsize=10,verticalalignment='center')
                    # move axis ticks off the key axis
                    key_ax.set_xticks([])
                    key_ax.set_yticks([])
                    

                    # save figure
                    fname = path_to_figures+core+'/'+core+'-'+section+'.png'
                    fig.savefig(fname,bbox_inches='tight')

                    # close figure to save memory
                    plt.close(fig)

                    # update progress bar
                    section_progress.update(1)

                    
        


In [21]:
def plot_tr(data,dmin,dmax,ACrescale,DCrescale):

    # make figure
    fig, axs = plt.subplots(1, 5, gridspec_kw={'width_ratios': [2, 3,2, 2, 3]},figsize=(9,6),dpi=200)

    # assign faces
    AC_t = None
    AC_r = None
    DC_t = None
    DC_r = None
    #loop through data 
    for d in data:
        if d.ACorDC == 'AC':
            if d.face == 't' or d.face == 'tr':
                AC_t = d
            if d.face == 'r':
                AC_r = d                    
        else:
            if d.face == 't' or d.face == 'tr':
                DC_t = d
            if d.face == 'r':
                DC_r = d

        
    # right-specific
    for a in [axs[0],axs[3]]:
        #a.yaxis.tick_right()
        a.set_xlim([60, 0])
        
    # top specific
    for a in [axs[1],axs[4]]:
        a.yaxis.tick_right()
        a.yaxis.set_label_position("right")
        a.set_xlim([0,120])
        
    # applies to all
    for a in [axs[0],axs[1],axs[3],axs[4]]:
        a.set_ylabel('Depth (m)')
        a.set_xlabel('Distance From Center (mm)',fontsize=6)
        a.set_ylim([dmax, dmin])

    for a,data_face in zip([axs[0],axs[1],axs[3],axs[4]],[AC_r,AC_t,DC_r,DC_t]):
        
        if data_face != None and len(data_face.y_vec) > 1:
            if data_face.face == 'r':
                yall = data_face.y_s - data_face.y_left
                yvec = data_face.y_vec - data_face.y_left
            else:
                yall = data_face.y_right - data_face.y_s
                yvec = data_face.y_right - data_face.y_vec
            
            if data_face.ACorDC =='AC':
                rescale = ACrescale
            else:
                rescale = DCrescale
        
        
            # plot data
            plotquarter(yvec,
                        yall,
                        data_face.depth_s,
                        data_face.meas_s,
                        data_face.surface_imperfection_s,
                        data_face.tephra_or_layer_s,
                        data_face.rock_or_debris_s,
                        a,
                        rescale)

    return fig, axs

In [22]:
def plot_tl(data,dmin,dmax,ACrescale,DCrescale):

    # make figure
    fig, axs = plt.subplots(1, 5, gridspec_kw={'width_ratios': [3, 3,2, 3, 3]},figsize=(9,6),dpi=200)

    # set data to empty
    AC_t = None
    AC_l = None
    DC_t = None
    DC_l = None
    #loop through data 
    for d in data:
        if d.ACorDC == 'AC':
            if d.face == 't' or d.face == 'tl':
                AC_t = d
            if d.face == 'l':
                AC_l = d                    
        else:
            if d.face == 't' or d.face == 'tl':
                DC_t = d
            if d.face == 'l':
                DC_l = d

    # top-specific
    for a in [axs[0],axs[3]]:
        #a.yaxis.tick_right()
        a.set_xlim([120, 0])
        
    # right specific
    for a in [axs[1],axs[4]]:
        a.yaxis.tick_right()
        a.yaxis.set_label_position("right")
        a.set_xlim([0,120])
        
    # applies to all
    for a in [axs[0],axs[1],axs[3],axs[4]]:
        a.set_ylabel('Depth (m)')
        a.set_xlabel('Distance From Center (mm)',fontsize=6)
        a.set_ylim([dmax, dmin])

    for a,data_face in zip([axs[1],axs[0],axs[4],axs[3]],[AC_l,AC_t,DC_l,DC_t]):
        
        if data_face != None and len(data_face.y_vec) > 1:
            if data_face.face == 'l':
                yall = data_face.y_right - data_face.y_s
                yvec = data_face.y_right -  data_face.y_vec
            else:
                yall = data_face.y_s -  data_face.y_left
                yvec =data_face.y_vec -  data_face.y_left
            
            if data_face.ACorDC =='AC':
                rescale = ACrescale
            else:
                rescale = DCrescale
        
        
             # plot data
            plotquarter(yvec,
                        yall,
                        data_face.depth_s,
                        data_face.meas_s,
                        data_face.surface_imperfection_s,
                        data_face.tephra_or_layer_s,
                        data_face.rock_or_debris_s,
                        a,
                        rescale)

    return fig, axs

In [23]:
def plot_w1w2(data,dmin,dmax,ACrescale,DCrescale):

    # make figure
    fig, axs = plt.subplots(1, 5, gridspec_kw={'width_ratios': [3, 3,3, 3, 3]},figsize=(9,6),dpi=100)

    # assign faces
    AC_w1 = None
    AC_w2 = None
    DC_w1 = None
    DC_w2 = None
    for d in data:
        if d.ACorDC == 'AC':
            if d.face == 'w1':
                AC_w1 = d
            if d.face == 'w2':
                AC_w2 = d
        else:
            if d.face == 'w1':
                DC_w1 = d
            if d.face == 'w2':
                DC_w2 = d

    # right-specific
        for a in [axs[0],axs[3]]:
            #a.yaxis.tick_right()
            a.set_xlim([55, 0])

    # top specific
    for a in [axs[1],axs[4]]:
        a.yaxis.tick_right()
        a.yaxis.set_label_position("right")
        a.set_xlim([0,55])
        
    # applies to all
    for a in [axs[0],axs[1],axs[3],axs[4]]:
        a.set_ylabel('Depth (m)')
        a.set_xlabel('Distance From Center (mm)',fontsize=6)
        a.set_ylim([dmax, dmin])

    for a,data_face in zip([axs[0],axs[1],axs[3],axs[4]],[AC_w2,AC_w1,DC_w2,DC_w1]):
        
        if data_face != None and len(data_face.y_vec) > 1:
            if data_face.face == 'w2':
                yall = data_face.y_s - data_face.y_left
                yvec = data_face.y_vec - data_face.y_left
            else:
                yall = data_face.y_right - data_face.y_s
                yvec = data_face.y_right - data_face.y_vec
            
            if data_face.ACorDC =='AC':
                rescale = ACrescale
            else:
                rescale = DCrescale
        
            # plot data
            plotquarter(yvec,
                        yall,
                        data_face.depth_s,
                        data_face.meas_s,
                        data_face.surface_imperfection_s,
                        data_face.tephra_or_layer_s,
                        data_face.rock_or_debris_s,
                        a,
                        rescale)

    return fig, axs


In [24]:
def plot_w1(data,dmin,dmax,ACrescale,DCrescale):

    # Make figure
    fig, axs = plt.subplots(1, 3, gridspec_kw={'width_ratios': [3,1, 3]},figsize=(9,6),dpi=100)

    # Assign faces
    AC_w1 = None
    DC_w1 = None
    for d in data:
        if d.ACorDC == 'AC':
            if d.face == 'w1':
                AC_w1 = d
        else:
            if d.face == 'w1':
                DC_w1 = d

    
    for a in [axs[0],axs[2]]:
        a.set_xlim([55,0])
        a.set_xlabel('Distance From O Line (mm)',fontsize=6)
        a.set_ylim([dmax, dmin])
        a.set_ylabel('Depth (m)')
        
        
    for a,data_face in zip([axs[0],axs[2]],[AC_w1,DC_w1]):
        
        if data_face != None and len(data_face.y_vec) > 1:

            # yall = data_face.y_right - data_face.y_s
            # yvec = data_face.y_right - data_face.y_vec

            # NEED TO CONIFRM THIS
            yall = data_face.y_s - data_face.y_left
            yvec = data_face.y_vec - data_face.y_left
            
            if data_face.ACorDC =='AC':
                rescale = ACrescale
            else:
                rescale = DCrescale
        
            # plot data
            plotquarter(yvec,
                        yall,
                        data_face.depth_s,
                        data_face.meas_s,
                        data_face.surface_imperfection_s,
                        data_face.tephra_or_layer_s,
                        data_face.rock_or_debris_s,
                        a,
                        rescale)


    # housekeeping
    axs[1].axis('off')
    axs[0].set_title('AC - W1')
    axs[2].set_title('DC - W1')

    return fig, axs

In [ ]:
plot_basic('ALHIC2501')

../../data/


ALHIC2501: starting:   0%|          | 0/189 [00:00<?, ?section/s]

In [ ]:
plot_basic('ALHIC2502')

../../data/


ALHIC2502: starting:   0%|          | 0/33 [00:00<?, ?section/s]

In [ ]:
plot_basic('ALHIC2201')

../../data/


ALHIC2201: starting:   0%|          | 0/23 [00:00<?, ?section/s]

/var/folders/b3/ghp5pyzj7_x9mhrs3x202bm80000gn/T/ipykernel_33721/3621879331.py:92: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


<Figure size 640x480 with 0 Axes>

In [ ]:
plot_basic('ALHIC2301')

../../data/


ALHIC2301: starting:   0%|          | 0/41 [00:00<?, ?section/s]

In [ ]:
plot_basic('ALHIC2401')

../../data/


ALHIC2401: starting:   0%|          | 0/39 [00:00<?, ?section/s]

In [ ]:
plot_basic('ALHIC2302')

../../data/


ALHIC2302: starting:   0%|          | 0/70 [00:00<?, ?section/s]

/var/folders/b3/ghp5pyzj7_x9mhrs3x202bm80000gn/T/ipykernel_33721/3621879331.py:92: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
/var/folders/b3/ghp5pyzj7_x9mhrs3x202bm80000gn/T/ipykernel_33721/3621879331.py:92: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
/var/folders/b3/ghp5pyzj7_x9mhrs3x202bm80000gn/T/ipykernel_33721/3621879331.py:92: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


<Figure size 640x480 with 0 Axes>